In [1]:
from pathlib import Path

soc_path = Path("00004-00000001.soc")
soc_path

PosixPath('00004-00000001.soc')

In [2]:
with soc_path.open("r", encoding="utf-8", errors="replace") as file:
    for i in range(10):
        line = file.readline()
        if not line:
            break
        print(line.rstrip("\n"))

# FILE NAME: 00004-00000001.soc
# TITLE: Netflix Prize Data
# DESCRIPTION: 
# DATA TYPE: soc
# MODIFICATION TYPE: induced
# RELATES TO: 
# RELATED FILES: 
# PUBLICATION DATE: 2013-08-17
# MODIFICATION DATE: 2022-09-16
# NUMBER ALTERNATIVES: 3


In [3]:
header_lines = []
data_lines = []

with soc_path.open("r", encoding="utf-8", errors="replace") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue
        if line.startswith("#"):
            header_lines.append(line)
        else:
            data_lines.append(line)

print("Number of header lines:", len(header_lines))
print("Number of data lines:", len(data_lines))

print("\nFirst 3 header lines:")
for h in header_lines[:3]:
    print(h)

print("\nFirst 3 data lines:")
for d in data_lines[:3]:
    print(d)

Number of header lines: 15
Number of data lines: 6

First 3 header lines:
# FILE NAME: 00004-00000001.soc
# TITLE: Netflix Prize Data
# DESCRIPTION:

First 3 data lines:
263: 2,1,3
249: 1,2,3
78: 1,3,2


In [4]:
title = None

for line in header_lines:
    if line.startswith("# TITLE:"):
        title = line[len("# TITLE:"):].strip()
        break
title

'Netflix Prize Data'

In [5]:
num_alternatives = None

for line in header_lines:
    if line.startswith("# NUMBER ALTERNATIVES:"):
        value = line[len("# NUMBER ALTERNATIVES:"):].strip()
        num_alternatives = int(value)
        break

num_alternatives

3

In [6]:
alternatives = {}

for line in header_lines:
    if line.startswith("# ALTERNATIVE NAME"):
        rest = line[len("# ALTERNATIVE NAME"):].strip()
        number_part, name_part = rest.split(":", 1)
        alt_id = int(number_part.strip())
        alt_name = name_part.strip()
        alternatives[alt_id] = alt_name

alternatives

{1: 'Shrek (Full-screen)', 2: 'The X-Files: Season 2', 3: 'The Punisher'}

In [7]:
rankings = []
for line in data_lines:
    count_part, order_part = line.split(":", 1)
    count = int(count_part.strip())
    order = [int(x.strip()) for x in order_part.split(",")]
    rankings.append((count, order))

print("Parsed rankings:", len(rankings))
print("First 3 parsed tuples:")
rankings[:3]

Parsed rankings: 6
First 3 parsed tuples:


[(263, [2, 1, 3]), (249, [1, 2, 3]), (78, [1, 3, 2])]

In [8]:
DO_NS = "http://www.semanticweb.org/decision-ontology#"
DATA_NS = "http://www.semanticweb.org/decision-ontology/data/preflib/"

ttl = (
    f"@prefix do: <{DO_NS}> .\n"
    f"@prefix data: <{DATA_NS}> .\n"
    "@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .\n"
    "@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .\n"
    "@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .\n\n"
)

print(ttl)

@prefix do: <http://www.semanticweb.org/decision-ontology#> .
@prefix data: <http://www.semanticweb.org/decision-ontology/data/preflib/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .




In [9]:
dataset_id = soc_path.stem
context_iri = f"data:context/{dataset_id}"

ttl += f"{context_iri} a do:Context ;\n"
ttl += f'   rdfs:label "{title}" .\n\n'

In [10]:
option_iris = {alt_id: f"data:option/{dataset_id}/{alt_id}" for alt_id in sorted(alternatives)}

In [11]:
options_list = ", ".join(option_iris[alt_id] for alt_id in sorted(option_iris))
ttl += f"{context_iri} do:hasAvailableOption {options_list} .\n\n"

In [12]:
for alt_id in sorted(alternatives):
    opt_iri = option_iris[alt_id]
    opt_label = alternatives[alt_id]
    ttl += f'{opt_iri} a do:Option ; rdfs:label "{opt_label}" . \n'

ttl += "\n"

In [13]:
print("\n".join(ttl.splitlines()[-20:]))

@prefix do: <http://www.semanticweb.org/decision-ontology#> .
@prefix data: <http://www.semanticweb.org/decision-ontology/data/preflib/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

data:context/00004-00000001 a do:Context ;
   rdfs:label "Netflix Prize Data" .

data:context/00004-00000001 do:hasAvailableOption data:option/00004-00000001/1, data:option/00004-00000001/2, data:option/00004-00000001/3 .

data:option/00004-00000001/1 a do:Option ; rdfs:label "Shrek (Full-screen)" . 
data:option/00004-00000001/2 a do:Option ; rdfs:label "The X-Files: Season 2" . 
data:option/00004-00000001/3 a do:Option ; rdfs:label "The Punisher" . 



In [18]:
for ballot_index, (weight, order) in enumerate(rankings, start=1):
    ballot_iri = f"data:ballot/{dataset_id}/{ballot_index}"
    bo_iris = [f"data:bo/{dataset_id}/{ballot_index}/{alt_id}" for alt_id in order]
    agent_iri = f"data:agent/{dataset_id}/{ballot_index}"

    ttl += (
    f"{ballot_iri} a do:Ballot, do:RankedBallot, do:CompleteBallot, do:AggregatedBallot ;\n"
    f"  do:hasContext {context_iri} ;\n"
    f"  do:hasWeight \"{weight}\"^^xsd:int ;\n"
    f"  do:hasBallotOption {', '.join(bo_iris)} .\n\n"
    )

    for rank_pos, alt_id in enumerate(order, start=1):
        bo_iri = f"data:bo/{dataset_id}/{ballot_index}/{alt_id}"
        opt_iri = option_iris[alt_id]
        ttl += (
            f"{bo_iri} a do:BallotOption ;\n"
            f"  do:refersToOption {opt_iri} ;\n"
            f"  do:optionRank \"{rank_pos}\"^^xsd:int .\n\n"
        )

    ttl += (
        f"{agent_iri} a do:Agent ;\n"
        f'  rdfs:label "Synthetic agent {ballot_index} ({dataset_id})" ;\n'
        f"  do:hasExpressed {ballot_iri} .\n\n"
    )

In [20]:
out_path = soc_path.with_suffix(".ttl")
with out_path.open("w", encoding="utf-8") as file:
    file.write(ttl)
out_path

PosixPath('00004-00000001.ttl')